# Cybersecurity Attack Detection - Model Training

This notebook demonstrates how to train and evaluate machine learning models for cybersecurity attack detection using our framework.

## Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

# Import our framework modules
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '..'))

from src.core.preprocessing import DataPreprocessor
from src.models.detector import CyberAttackDetector, ModelComparer, ModelRegistry
from src.utils.helpers import create_sample_dataset, calculate_dataset_statistics, PerformanceTimer

# Configure plotting
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 8)

print("Framework modules imported successfully!")

## Data Loading and Exploration

In [ ]:
# Create sample dataset for demonstration
print("Creating sample cybersecurity dataset...")
dataset = create_sample_dataset(n_samples=2000, n_features=15, n_classes=2)

print(f"Dataset shape: {dataset.shape}")
print(f"\nColumns: {list(dataset.columns)}")
print(f"\nLabel distribution:")
print(dataset['label'].value_counts())

dataset.head()

In [ ]:
# Calculate comprehensive dataset statistics
stats = calculate_dataset_statistics(dataset)

print("Dataset Statistics:")
print(f"Shape: {stats['shape']}")
print(f"Memory usage: {stats['memory_usage_mb']:.2f} MB")
print(f"Missing values: {stats['missing_values']}")
print(f"Duplicate rows: {stats['duplicate_rows']}")
print(f"\nData types distribution:")
for dtype, count in stats['dtypes'].items():
    print(f"  {dtype}: {count} columns")

In [ ]:
# Visualize label distribution
plt.figure(figsize=(10, 6))

plt.subplot(1, 2, 1)
dataset['label'].value_counts().plot(kind='bar')
plt.title('Label Distribution')
plt.xlabel('Class')
plt.ylabel('Count')

plt.subplot(1, 2, 2)
dataset['protocol_type'].value_counts().plot(kind='pie', autopct='%1.1f%%')
plt.title('Protocol Type Distribution')

plt.tight_layout()
plt.show()

## Data Preprocessing

In [ ]:
# Initialize preprocessor and run full pipeline
preprocessor = DataPreprocessor()

with PerformanceTimer("Data preprocessing"):
    X, y, validation_results = preprocessor.full_preprocessing_pipeline(dataset)

print("Preprocessing completed!")
print(f"Features shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"\nValidation results: {validation_results['is_valid']}")

if validation_results['warnings']:
    print(f"Warnings: {validation_results['warnings']}")

In [ ]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
X_train_scaled, X_test_scaled = preprocessor.scale_features(X_train, X_test)

print(f"Training set shape: {X_train_scaled.shape}")
print(f"Test set shape: {X_test_scaled.shape}")
print(f"Training labels distribution:")
print(pd.Series(y_train).value_counts())

## Model Training and Evaluation

In [ ]:
# Show available models
available_models = ModelRegistry.get_available_models()
print("Available models:")
for model_name in available_models.keys():
    print(f"  - {model_name}")

In [ ]:
# Train a Random Forest model
print("Training Random Forest model...")
rf_detector = CyberAttackDetector('random_forest')

with PerformanceTimer("Random Forest training"):
    training_results = rf_detector.train(
        X_train_scaled, y_train, feature_names=X.columns.tolist()
    )

print(f"Training completed!")
print(f"CV Accuracy: {training_results['cv_mean']:.4f} (+/- {training_results['cv_std']*2:.4f})")
print(f"Training time: {training_results['training_time']:.2f} seconds")

In [ ]:
# Evaluate the trained model
evaluation_results = rf_detector.evaluate(X_test_scaled, y_test)

print("Evaluation Results:")
print(f"Accuracy: {evaluation_results['accuracy']:.4f}")
print(f"Precision: {evaluation_results['precision']:.4f}")
print(f"Recall: {evaluation_results['recall']:.4f}")
print(f"F1 Score: {evaluation_results['f1_score']:.4f}")

if 'auc_score' in evaluation_results:
    print(f"AUC Score: {evaluation_results['auc_score']:.4f}")

In [ ]:
# Visualize confusion matrix
plt.figure(figsize=(8, 6))
cm = np.array(evaluation_results['confusion_matrix'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Random Forest')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

print("\nClassification Report:")
print(evaluation_results['classification_report'])

In [ ]:
# Visualize feature importance
if 'feature_importance' in evaluation_results:
    feature_importance = pd.Series(evaluation_results['feature_importance'])
    
    plt.figure(figsize=(12, 8))
    feature_importance.head(15).plot(kind='barh')
    plt.title('Top 15 Feature Importances - Random Forest')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()

## Model Comparison

In [ ]:
# Compare multiple models
print("Comparing multiple models...")
comparer = ModelComparer()

# Add models to compare
comparer.add_model('Random Forest', 'random_forest')
comparer.add_model('Logistic Regression', 'logistic_regression')
comparer.add_model('Neural Network', 'neural_network')

# Run comparison
with PerformanceTimer("Model comparison"):
    comparison_results = comparer.compare_models(
        X_train_scaled, y_train, X_test_scaled, y_test, X.columns.tolist()
    )

In [ ]:
# Display comparison summary
summary = comparison_results['summary']

print("Model Comparison Summary:")
print(f"Best Accuracy: {summary['best_accuracy']['model']} ({summary['best_accuracy']['score']:.4f})")
print(f"Best Precision: {summary['best_precision']['model']} ({summary['best_precision']['score']:.4f})")
print(f"Best Recall: {summary['best_recall']['model']} ({summary['best_recall']['score']:.4f})")
print(f"Best F1 Score: {summary['best_f1']['model']} ({summary['best_f1']['score']:.4f})")
print(f"Fastest Training: {summary['fastest_training']['model']} ({summary['fastest_training']['time']:.2f}s)")

print("\nAccuracy Ranking:")
for i, (model, accuracy) in enumerate(summary['accuracy_ranking'], 1):
    print(f"{i}. {model}: {accuracy:.4f}")

In [ ]:
# Visualize model comparison
detailed_results = comparison_results['detailed_results']

# Extract metrics for visualization
models = list(detailed_results.keys())
metrics = ['accuracy', 'precision', 'recall', 'f1_score']

comparison_data = pd.DataFrame(index=models, columns=metrics)
for model in models:
    eval_results = detailed_results[model]['evaluation']
    for metric in metrics:
        comparison_data.loc[model, metric] = eval_results[metric]

comparison_data = comparison_data.astype(float)

# Plot comparison
plt.figure(figsize=(12, 8))
comparison_data.plot(kind='bar', rot=45)
plt.title('Model Performance Comparison')
plt.xlabel('Models')
plt.ylabel('Score')
plt.legend(title='Metrics')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

## Hyperparameter Tuning

In [ ]:
# Demonstrate hyperparameter tuning
print("Performing hyperparameter tuning for Random Forest...")

tuned_detector = CyberAttackDetector('random_forest')

with PerformanceTimer("Hyperparameter tuning"):
    tuning_results = tuned_detector.hyperparameter_tuning(
        X_train_scaled, y_train, cv_folds=3  # Reduced for demo
    )

print(f"Tuning completed!")
print(f"Best parameters: {tuning_results['best_params']}")
print(f"Best CV score: {tuning_results['best_score']:.4f}")
print(f"Combinations tested: {tuning_results['n_combinations']}")

In [ ]:
# Evaluate tuned model
tuned_evaluation = tuned_detector.evaluate(X_test_scaled, y_test)

print("Tuned Model Evaluation:")
print(f"Accuracy: {tuned_evaluation['accuracy']:.4f}")
print(f"Precision: {tuned_evaluation['precision']:.4f}")
print(f"Recall: {tuned_evaluation['recall']:.4f}")
print(f"F1 Score: {tuned_evaluation['f1_score']:.4f}")

# Compare with default model
print(f"\nImprovement over default model:")
print(f"Accuracy: {tuned_evaluation['accuracy'] - evaluation_results['accuracy']:.4f}")
print(f"F1 Score: {tuned_evaluation['f1_score'] - evaluation_results['f1_score']:.4f}")

## Model Saving and Loading

In [ ]:
# Save the best model
import os
os.makedirs('../models', exist_ok=True)

model_path = '../models/best_rf_model.pkl'
tuned_detector.save_model(model_path)
print(f"Model saved to {model_path}")

# Demonstrate loading
loaded_detector = CyberAttackDetector('random_forest')
loaded_detector.load_model(model_path)
print("Model loaded successfully!")

# Verify loaded model works
loaded_predictions = loaded_detector.predict(X_test_scaled)
original_predictions = tuned_detector.predict(X_test_scaled)

print(f"Predictions match: {np.array_equal(loaded_predictions, original_predictions)}")

## Conclusion

This notebook demonstrated:

1. **Data Loading and Exploration**: How to load and explore cybersecurity datasets
2. **Data Preprocessing**: Complete preprocessing pipeline with validation
3. **Model Training**: Training different types of models
4. **Model Evaluation**: Comprehensive evaluation with multiple metrics
5. **Model Comparison**: Comparing multiple models side-by-side
6. **Hyperparameter Tuning**: Optimizing model performance
7. **Model Persistence**: Saving and loading trained models

The framework provides a robust, modular approach to cybersecurity attack detection with built-in best practices for data preprocessing, model management, and evaluation.